# Modrinth Project Version Ingestion

This notebook performs only the detailed Modrinth version pull.

Flow:

1. Read `project_id` and `project_type` from Silver `base_api_project_listings`.
2. Fetch `/project/{project_id}/version` asynchronously for every project.
3. Preserve the complete version response as raw JSON in Bronze — no flattening or field selection happens here.
4. Write one Bronze row per project per run while updating console progress in real time.

The project listing/search API is intentionally not called here. Silver defines the project universe for this job.

In [ ]:
import os
import duckdb
import httpx
import json
import asyncio
from datetime import datetime, timezone
import requests
import uuid
import polars as pl
import time
import pandas as pd
import yaml
from config import Bundle

## Progress and rate limiting

The version endpoint requires one request per project, so request pacing matters much more here than it did for paginated project listings.

The limiter keeps asynchronous concurrency while spacing request starts to stay near Modrinth's 300-request-per-minute limit. A `429` pauses the shared limiter for exactly 61 seconds.

In [2]:
class IngestionProgress:
    """Mutable progress shared by concurrent version-fetch tasks."""

    def __init__(self, total_projects: int, project_type: str):
        self.total_projects = total_projects
        self.project_type = project_type
        self.completed_projects = 0
        self.failed_projects = 0
        self.versions_fetched = 0
        self.project_payloads_written = 0
        self.start_time = time.perf_counter()
        self.last_print_time = 0.0
        self.lock = asyncio.Lock()


class ModrinthRateLimiter:
    """Pace request starts while still allowing requests to overlap in flight."""

    def __init__(
        self,
        max_concurrency: int = 8,
        requests_per_minute: int = 300,
        full_reset_seconds: float = 61.0,
    ):
        self.semaphore = asyncio.Semaphore(max_concurrency)
        self.request_interval_seconds = 60.0 / requests_per_minute
        self.full_reset_seconds = full_reset_seconds

        self.next_request_time = 0.0
        self.pause_until = 0.0

        self.request_lock = asyncio.Lock()
        self.pause_lock = asyncio.Lock()

    async def wait_if_paused(self) -> None:
        while True:
            async with self.pause_lock:
                wait_seconds = self.pause_until - time.monotonic()

            if wait_seconds <= 0:
                return

            await asyncio.sleep(wait_seconds)

    async def wait_for_request_slot(self) -> None:
        # Serializing request start times prevents a large async batch from
        # bursting through the API's requests-per-minute allowance.
        async with self.request_lock:
            now = time.monotonic()
            wait_seconds = self.next_request_time - now

            if wait_seconds > 0:
                await asyncio.sleep(wait_seconds)

            self.next_request_time = (
                time.monotonic() + self.request_interval_seconds
            )

    async def pause_for_rate_limit(self) -> None:
        # All tasks share the same pause window after a 429 response.
        async with self.pause_lock:
            new_pause_until = (
                time.monotonic() + self.full_reset_seconds
            )
            self.pause_until = max(
                self.pause_until,
                new_pause_until,
            )

In [3]:
async def print_progress(
    progress: IngestionProgress,
    force: bool = False,
) -> None:
    """Refresh one console line with current ingestion pacing."""
    now = time.perf_counter()

    # Limit console refreshes so printing does not become a bottleneck.
    if not force and (now - progress.last_print_time) < 0.25:
        return

    elapsed_seconds = max(
        now - progress.start_time,
        0.001,
    )

    percent_complete = (
        progress.completed_projects
        / progress.total_projects
        * 100
        if progress.total_projects
        else 0.0
    )

    requests_per_minute = (
        progress.completed_projects
        / elapsed_seconds
        * 60
    )

    print(
        f"\r\t(INFO) "
        f"projects={progress.completed_projects:,}/{progress.total_projects:,} "
        f"({percent_complete:6.2f}%) | "
        f"versions={progress.versions_fetched:,} | "
        f"payloads_written={progress.project_payloads_written:,} | "
        f"failed={progress.failed_projects:,} | "
        f"pace={requests_per_minute:,.1f} req/min",
        end="",
        flush=True,
    )

    progress.last_print_time = now


async def mark_project_complete(
    progress: IngestionProgress,
    versions_fetched: int,
    failed: bool = False,
) -> None:
    async with progress.lock:
        progress.completed_projects += 1
        progress.versions_fetched += versions_fetched

        if failed:
            progress.failed_projects += 1

        await print_progress(progress)


async def mark_payloads_written(
    progress: IngestionProgress,
    payloads_written: int,
) -> None:
    async with progress.lock:
        progress.project_payloads_written += payloads_written
        await print_progress(progress)

## HTTP helper

All API requests use one shared `httpx.AsyncClient` and one shared rate limiter for the entire run.

In [4]:
async def _get_json_with_backoff(
    client: httpx.AsyncClient,
    url: str,
    rate_limiter: ModrinthRateLimiter,
    params: dict | None = None,
    max_retries: int = 6,
) -> dict | list:
    last_error = None

    for _ in range(max_retries):
        await rate_limiter.wait_if_paused()

        try:
            # Pace request starts while still allowing concurrent in-flight requests.
            await rate_limiter.wait_for_request_slot()

            async with rate_limiter.semaphore:
                response = await client.get(url, params=params)

            if response.status_code == 429:
                await rate_limiter.pause_for_rate_limit()
                continue

            response.raise_for_status()
            return response.json()

        except (
            httpx.ReadTimeout,
            httpx.ConnectTimeout,
            httpx.RemoteProtocolError,
            httpx.NetworkError,
        ) as e:
            last_error = e
            await asyncio.sleep(rate_limiter.full_reset_seconds)

        except httpx.HTTPStatusError as e:
            last_error = e

            if 500 <= e.response.status_code < 600:
                await asyncio.sleep(rate_limiter.full_reset_seconds)
                continue

            raise

    raise RuntimeError(
        f"Exceeded retries for {url}. Last error: {last_error}"
    )

## Read the project universe from Silver

The detailed ingestion only needs `project_id` and `project_type`. Project titles, slugs, loaders, files, dependencies, and other details remain inside the version API payload and are intentionally not modeled in Bronze.

In [5]:
def read_base_projects(b1: Bundle) -> pl.DataFrame:
    """Read the distinct projects that should receive a version API call."""
    with duckdb.connect(
        b1.silver_db_path,
        read_only=True,
    ) as silver_con:
        table_columns = {
            row[0]
            for row in silver_con.execute(
                f"DESCRIBE {b1.base_api_project_listings_table_name}"
            ).fetchall()
        }

        required_columns = {
            "project_id",
            "project_type",
        }
        missing_columns = required_columns - table_columns

        if missing_columns:
            raise ValueError(
                "base_api_project_listings is missing required columns: "
                f"{sorted(missing_columns)}"
            )

        projects_df = silver_con.execute(
            f"""
            SELECT DISTINCT
                project_id,
                project_type
            FROM {b1.base_api_project_listings_table_name}
            WHERE project_id IS NOT NULL
              AND project_type IS NOT NULL
            ORDER BY
                project_type,
                project_id
            """
        ).pl()

    return projects_df


def validate_version_payload_table_schema(b1: Bundle) -> None:
    """
    Fail early if an older flattened Bronze version table still exists.

    `CREATE TABLE IF NOT EXISTS` does not migrate an existing DuckDB table,
    so this check prevents a long API run from failing only when it begins
    writing data.
    """
    expected_columns = {
        "run_id",
        "project_type",
        "project_id",
        "payload",
        "c_pull_timestamp_utc",
    }

    with duckdb.connect(
        b1.bronze_db_path,
        read_only=True,
    ) as bronze_con:
        actual_columns = {
            row[0]
            for row in bronze_con.execute(
                f"DESCRIBE {b1.raw_api_file_tables_table_name}"
            ).fetchall()
        }

    if actual_columns != expected_columns:
        raise ValueError(
            f"{b1.raw_api_file_tables_table_name} does not match the "
            "raw JSON payload schema. "
            f"Expected columns: {sorted(expected_columns)}. "
            f"Found: {sorted(actual_columns)}. "
            "Update config.py, drop the old table once, and rerun "
            "b1.init_db('bronze')."
        )

## Fetch raw project version payloads

Each project produces one API request. The complete list returned by Modrinth is preserved unchanged as that project's Bronze JSON payload.

No loader expansion, primary-file selection, dependency serialization, or datatype enforcement occurs here. Those transformations belong in Bronze-to-Silver processing.

In [6]:
async def fetch_project_versions(
    b1: Bundle,
    client: httpx.AsyncClient,
    rate_limiter: ModrinthRateLimiter,
    project: dict,
    progress: IngestionProgress,
) -> dict:
    """Fetch and return the complete version payload for one project."""
    project_id = project["project_id"]

    try:
        versions = await _get_json_with_backoff(
            client=client,
            url=f"{b1.modrinth_base_url}/project/{project_id}/version",
            rate_limiter=rate_limiter,
            params={"include_changelog": "false"},
        )

        if not isinstance(versions, list):
            raise TypeError(
                f"Expected a list of versions for {project_id}, "
                f"got {type(versions).__name__}"
            )

        await mark_project_complete(
            progress=progress,
            versions_fetched=len(versions),
        )

        return {
            "project_type": project["project_type"],
            "project_id": project_id,
            "payload": versions,
            "c_pull_timestamp_utc": datetime.now(timezone.utc),
            "error": None,
        }

    except Exception as e:
        await mark_project_complete(
            progress=progress,
            versions_fetched=0,
            failed=True,
        )

        return {
            "project_type": project["project_type"],
            "project_id": project_id,
            "payload": [],
            "c_pull_timestamp_utc": None,
            "error": e,
        }

## Build and write Bronze payload rows

One successful project API response becomes exactly one Bronze row:

- `run_id`
- `project_type`
- `project_id`
- complete `payload` JSON array
- `c_pull_timestamp_utc`

Expected `Bundle` schema:

```sql
CREATE TABLE IF NOT EXISTS raw_api_file_tables (
    run_id                VARCHAR NOT NULL,
    project_type          VARCHAR NOT NULL,
    project_id            VARCHAR NOT NULL,
    payload               JSON NOT NULL,
    c_pull_timestamp_utc  TIMESTAMPTZ NOT NULL,
    PRIMARY KEY (run_id, project_type, project_id)
);
```

The table grain is one project-version payload per project per ingestion run. The raw response is preserved so Silver can later infer and normalize version fields without losing source data.

In [7]:
def build_version_payload_rows(
    run_id: str,
    results: list[dict],
) -> list[tuple]:
    """Convert successful API results into the five-column Bronze envelope."""
    rows = []

    for result in results:
        if result["error"] is not None:
            continue

        rows.append(
            (
                run_id,
                result["project_type"],
                result["project_id"],
                json.dumps(result["payload"]),
                result["c_pull_timestamp_utc"],
            )
        )

    return rows


def write_version_payload_rows(
    bronze_con,
    table_name: str,
    version_rows: list[tuple],
) -> int:
    """Write a batch of raw project-version JSON payloads to Bronze."""
    if not version_rows:
        return 0

    insert_sql = f"""
        INSERT OR REPLACE INTO {table_name}
        (
            run_id,
            project_type,
            project_id,
            payload,
            c_pull_timestamp_utc
        )
        VALUES (?, ?, ?, ?, ?)
    """

    bronze_con.executemany(
        insert_sql,
        version_rows,
    )

    return len(version_rows)

## Version ingestion orchestration

Projects are processed in bounded batches. Requests inside each batch are asynchronous and use the shared rate limiter. After a batch finishes, only successful raw payloads are written to DuckDB.

Progress distinguishes the number of version objects fetched from the number of project payload rows written to Bronze.

In [8]:
async def ingest_project_type_versions(
    b1: Bundle,
    run_id: str,
    project_type: str,
    projects_df: pl.DataFrame,
    client: httpx.AsyncClient,
    rate_limiter: ModrinthRateLimiter,
    bronze_con,
    project_batch_size: int = 250,
) -> dict:
    """Fetch and persist all version payloads for one project type."""
    project_type_df = projects_df.filter(
        pl.col("project_type") == project_type
    )

    projects = project_type_df.to_dicts()
    total_projects = len(projects)

    progress = IngestionProgress(
        total_projects=total_projects,
        project_type=project_type,
    )

    failed_project_ids = []
    start_time = time.perf_counter()

    await print_progress(
        progress,
        force=True,
    )

    for start_idx in range(
        0,
        total_projects,
        project_batch_size,
    ):
        project_batch = projects[
            start_idx:start_idx + project_batch_size
        ]

        results = await asyncio.gather(
            *[
                fetch_project_versions(
                    b1=b1,
                    client=client,
                    rate_limiter=rate_limiter,
                    project=project,
                    progress=progress,
                )
                for project in project_batch
            ]
        )

        failed_project_ids.extend(
            result["project_id"]
            for result in results
            if result["error"] is not None
        )

        version_rows = build_version_payload_rows(
            run_id=run_id,
            results=results,
        )

        payloads_written = write_version_payload_rows(
            bronze_con=bronze_con,
            table_name=b1.raw_api_file_tables_table_name,
            version_rows=version_rows,
        )

        await mark_payloads_written(
            progress=progress,
            payloads_written=payloads_written,
        )

    await print_progress(
        progress,
        force=True,
    )
    print()

    duration_seconds = time.perf_counter() - start_time

    if failed_project_ids:
        failed_preview = ", ".join(
            failed_project_ids[:10]
        )
        print(
            f"	(WARN) failed projects: {len(failed_project_ids):,} | "
            f"first IDs: {failed_preview}"
        )

    return {
        "run_id": run_id,
        "project_type": project_type,
        "projects_processed": progress.completed_projects,
        "projects_failed": progress.failed_projects,
        "versions_fetched": progress.versions_fetched,
        "project_payloads_written": progress.project_payloads_written,
        "duration_seconds": duration_seconds,
    }

In [ ]:
def start_ingestion_log(
    b1: Bundle,
    run_id: str,
    ingestion_type: str,
    project_type: str,
) -> None:

    insert_sql = f"""
        INSERT INTO {b1.ingestion_log_table_name}
        (
            run_id,
            ingestion_type,
            api_url,
            project_type,
            status,
            start_time
        )
        VALUES (?, ?, ?, ?, ?, CURRENT_TIMESTAMP)
    """

    with duckdb.connect(
        b1.bronze_db_path
    ) as bronze_con:

        bronze_con.execute(
            insert_sql,
            [
                run_id,
                ingestion_type,
                b1.modrinth_base_url,
                project_type,
                "running",
            ],
        )

In [ ]:
def finish_ingestion_log(
    b1: Bundle,
    run_id: str,
    ingestion_type: str,
    project_type: str,
    status: str,
    records_processed: int = 0,
    records_written: int = 0,
    records_failed: int = 0,
    records_skipped: int = 0,
    nested_records_fetched: int | None = None,
    failed_record_ids: list[str] | None = None,
    skipped_record_ids: list[str] | None = None,
    error_message: str | None = None,
) -> None:

    update_sql = f"""
        UPDATE {b1.ingestion_log_table_name}
        SET
            status = ?,
            records_processed = ?,
            records_written = ?,
            records_failed = ?,
            records_skipped = ?,
            nested_records_fetched = ?,
            failed_record_ids = ?,
            skipped_record_ids = ?,
            error_message = ?,
            end_time = CURRENT_TIMESTAMP,
            duration_seconds = EXTRACT(
                EPOCH FROM (
                    CURRENT_TIMESTAMP - start_time
                )
            )
        WHERE run_id = ?
          AND ingestion_type = ?
          AND project_type = ?
    """

    with duckdb.connect(
        b1.bronze_db_path
    ) as bronze_con:

        bronze_con.execute(
            update_sql,
            [
                status,
                records_processed,
                records_written,
                records_failed,
                records_skipped,
                nested_records_fetched,
                failed_record_ids,
                skipped_record_ids,
                error_message,
                run_id,
                ingestion_type,
                project_type,
            ],
        )

## Run ingestion

A new `run_id` is created for the full detailed-ingestion execution. Project types come directly from Silver, so this notebook never calls the general Modrinth search endpoint.

The Bronze version table is validated before any API work begins. If an older flattened version table still exists, drop it once after updating `config.py`, rerun `b1.init_db('bronze')`, and then run this notebook again.

For a test run, uncomment the `project_types = ["mod"]` line.

In [ ]:
# Initialize Bundle and make sure the Bronze target tables exist.
b1 = Bundle()

b1.build_layer_directory(
    b1.bronze_env_folder_path
)

b1.init_db("bronze")

# Fail before making API calls if the old flattened
# version table still exists.
validate_version_payload_table_schema(b1)

# Silver is the authoritative universe of projects
# for detailed/version ingestion.
base_projects_df = read_base_projects(b1)

project_types = (
    base_projects_df
    .select("project_type")
    .unique()
    .sort("project_type")
    .to_series()
    .to_list()
)

# TEMPORARY TEST OVERRIDE:
# project_types = ["mod"]


async def ingestion() -> pl.DataFrame:

    # One run_id represents the entire project-version ingestion run.
    run_id = str(uuid.uuid4())

    ingestion_type = "project_versions"

    summaries = []

    total_projects = base_projects_df.height

    print(
        f"[INFO] Starting detailed version ingestion | "
        f"run_id={run_id} | "
        f"projects={total_projects:,} | "
        f"project_types={len(project_types):,}"
    )

    # Shared rate limiter across all project types.
    rate_limiter = ModrinthRateLimiter(
        max_concurrency=b1.concurrency,
        requests_per_minute=300,
        full_reset_seconds=61,
    )

    timeout = httpx.Timeout(
        connect=10.0,
        read=60.0,
        write=10.0,
        pool=60.0,
    )

    limits = httpx.Limits(
        max_connections=max(
            10,
            b1.concurrency * 2,
        ),
        max_keepalive_connections=max(
            5,
            b1.concurrency,
        ),
    )

    async with httpx.AsyncClient(
        headers=b1.headers,
        timeout=timeout,
        limits=limits,
        follow_redirects=True,
    ) as client:

        # Keep one DuckDB writer connection open
        # for the entire version ingestion run.
        with duckdb.connect(
            b1.bronze_db_path
        ) as bronze_con:

            for idx, project_type in enumerate(project_types):

                print(
                    f"({idx + 1}/{len(project_types)}) "
                    f"Ingesting project versions: {project_type}"
                )

                # Create the log row before starting this project type.
                start_ingestion_log(
                    b1=b1,
                    run_id=run_id,
                    ingestion_type=ingestion_type,
                    project_type=project_type,
                )

                try:
                    summary = await ingest_project_type_versions(
                        b1=b1,
                        run_id=run_id,
                        project_type=project_type,
                        projects_df=base_projects_df,
                        client=client,
                        rate_limiter=rate_limiter,
                        bronze_con=bronze_con,
                    )

                    summaries.append(summary)

                    # --------------------------------------------------
                    # Map version-specific metrics to the generalized
                    # ingestion_log schema.
                    # --------------------------------------------------

                    records_processed = (
                        summary["projects_processed"]
                    )

                    records_written = (
                        summary["project_payloads_written"]
                    )

                    records_failed = (
                        summary["projects_failed"]
                    )

                    records_skipped = 0

                    nested_records_fetched = (
                        summary["versions_fetched"]
                    )

                    print(
                        f"\t(INFO) projects processed: "
                        f"{records_processed:,}"
                    )

                    print(
                        f"\t(INFO) project payloads written: "
                        f"{records_written:,}"
                    )

                    print(
                        f"\t(INFO) projects failed: "
                        f"{records_failed:,}"
                    )

                    print(
                        f"\t(INFO) versions fetched: "
                        f"{nested_records_fetched:,}"
                    )

                    # The project-type ingestion itself completed
                    # successfully. Individual API failures are tracked
                    # separately in records_failed.
                    finish_ingestion_log(
                        b1=b1,
                        run_id=run_id,
                        ingestion_type=ingestion_type,
                        project_type=project_type,
                        status="success",
                        records_processed=records_processed,
                        records_written=records_written,
                        records_failed=records_failed,
                        records_skipped=records_skipped,
                        nested_records_fetched=nested_records_fetched,
                        failed_record_ids=None,
                        skipped_record_ids=None,
                        error_message=None,
                    )

                except Exception as e:

                    # A project-type-level exception means this portion
                    # of the ingestion could not complete normally.
                    finish_ingestion_log(
                        b1=b1,
                        run_id=run_id,
                        ingestion_type=ingestion_type,
                        project_type=project_type,
                        status="failed",
                        error_message=str(e),
                    )

                    print(
                        f"\t(ERROR) Failed to ingest versions for "
                        f"project_type {project_type}: {e}"
                    )

                    # Continue with the remaining project types.
                    continue

    if not summaries:
        return pl.DataFrame()

    return pl.DataFrame(summaries)


version_ingestion_summary_df = await ingestion()

version_ingestion_summary_df

[INFO] Starting detailed version ingestion | run_id=9686352e-8206-416a-a615-74f4da8391aa | projects=157,757 | project_types=7
(1/7) Ingesting project versions: datapack
	(INFO) projects=14,165/14,165 (100.00%) | versions=94,120 | payloads_written=14,164 | failed=1 | pace=295.5 req/min
	(WARN) failed projects: 1 | first IDs: sOGdQI8B
(2/7) Ingesting project versions: minecraft_java_server
	(INFO) projects=2,016/2,016 (100.00%) | versions=3,632 | payloads_written=2,016 | failed=0 | pace=295.1 req/min
(3/7) Ingesting project versions: mod
	(INFO) projects=72,802/72,802 (100.00%) | versions=952,384 | payloads_written=72,787 | failed=15 | pace=291.3 req/min
	(WARN) failed projects: 15 | first IDs: 2hhxCwai, 3W8aUMr8, GJlOCKX5, HmE5CkZ9, N3rHSWYa, Yl6VUsst, fiSivfZY, hc50lg4D, mHjMssSV, mKEJM9ch
(4/7) Ingesting project versions: modpack
	(INFO) projects=18,119/18,119 (100.00%) | versions=167,093 | payloads_written=18,115 | failed=4 | pace=242.2 req/min
	(WARN) failed projects: 4 | first IDs:

run_id,project_type,projects_processed,projects_failed,versions_fetched,project_payloads_written,duration_seconds
str,str,i64,i64,i64,i64,f64
"""9686352e-8206-416a-a615-74f4da…","""datapack""",14165,1,94120,14164,2876.544429
"""9686352e-8206-416a-a615-74f4da…","""minecraft_java_server""",2016,0,3632,2016,409.856691
"""9686352e-8206-416a-a615-74f4da…","""mod""",72802,15,952384,72787,14996.706185
"""9686352e-8206-416a-a615-74f4da…","""modpack""",18119,4,167093,18115,4488.25915
"""9686352e-8206-416a-a615-74f4da…","""plugin""",16849,2,113858,16847,3504.086984
"""9686352e-8206-416a-a615-74f4da…","""resourcepack""",32972,5,130865,32967,6716.950382
"""9686352e-8206-416a-a615-74f4da…","""shader""",834,0,3767,834,169.920634
